In [5]:
from typing import Optional, Callable
import numpy as np

import interfere
import sdeint
from sdeint.integrate import _check_args
from sdeint.wiener import Ikpw, deltaW

In [ ]:
def _simulate_sri2(
    model,
    t: np.ndarray,
    prior_states: np.ndarray,
    prior_t: Optional[np.ndarray] = None,
    intervention: Optional[Callable[[np.ndarray, float], np.ndarray]] = None,
    rng: np.random.Generator = np.random.default_rng(),
    dW: Optional[np.ndarray] = None,
    I: Optional[np.ndarray] = None,
) -> np.ndarray:
    """Simulates intervened SDE with Strong RK2 (SRI2/SRS2).

    Args:
        t (ndarray): A (n,) array of the time points where the   
            dynamic model will be simulated. The first entry of `t` must
            equal the last entry of `prior_t`. If `prior_t` is None, then
            the values of `prior_t` will be assumed to be evenly spaced time
            values ending with the first entry of `t`. If `t` does not
            contain evenly spaced time values, then the inference of
            `prior_t` will throw an error.
        prior_states (ndarray): A (m,) or (p, m) array of the initial
            condition or the prior states of the system.
        prior_t (ndarray): A time array with shape (p,) corresponding to the
            rows of `prior_states`. The last entry of `prior_t` must equal
            the first entry of `t`. If `prior_t` is None, then
            the values of `prior_t` will be assumed to be evenly spaced time
            values ending with the first entry of `t`. If `t` does not
            contain evenly spaced time values, then the inference of
            `prior_t` will throw an error.
        intervention (callable): A function that accepts (1) a vector of the
            current state of the dynamic model and (2) the current time. It should return a modified state. The function will be used in the
            following way. If the dynamic model without the intervention can be described 
            as
            
            x(t+dt) = F(x(t))
            
            where dt is the timestep size, x(t) is the trajectory, and F is
            the function that uses the current state to compute the state at
            the next timestep. Then the intervention function will be used
            to simulate the system

                z(t+dt) = F(g(z(t), t), t)
                x_do(t) = g(z(t), t)

            where x_do is the trajectory of the intervened system and g is 
            the intervention function.
        rng (RandomState): A numpy random state for reproducibility. (Uses 
            numpy's mtrand random number generator by default.)
        dW: optional array of shape (len(time_points)-1, self.dim). This is
            for advanced use, if you want to use a specific realization of
            the independent Wiener processes. If not provided Wiener
            increments will be generated randomly.
        I: optional array of shape (len(tspan)-1, m, m).
            These optional arguments dW and I are for advanced use, if you 
            want to use a specific realization of the d independent Wiener 
            processes and their multiple integrals at each time step. If not provided, suitable values will be generated randomly.

    Returns:
        X (ndarray): An (n, m) array containing a realization of the   
            trajectory of the m dimensional system corresponding to the n
            times in `t`. The first row of X contains the last row of 
            `prior_states`.
    
        with optional exogenous intervention.

    - f = self.drift
    - G = self.noise
    - handles user‐provided dW and IJ (repeated integrals)
    - applies `intervention` before each step
    - adds measurement_noise_std after integration

    See Roessler 2010 for the tableau.
    """
    var_dict = {}
    # Set up initial state and dimensions.
    y0 = prior_states[-1, :].copy()
    if intervention is not None:
        y0 = intervention(y0, t[0])
    N = len(t)
    d = model.dim
    Y = np.zeros((N, d), dtype=y0.dtype)
    Y[0] = y0

    # Time step size. Assumed equally spaced.
    h = (t[N-1] - t[0]) / (N - 1)

    # Pre‐generate Wiener increments if needed.
    if dW is None:
        # shape (N-1, d)
        dW = rng.normal(0.0, np.sqrt(h), size=(N - 1, d))

    # Pre‐generate repeated integrals.
    if I is None:
        _, I = Ikpw(dW, h, generator=rng)

    # Define f and G wrappers.
    def f(y, tau):
        return model.drift(y, tau)

    def G(y, tau):
        return model.noise(y, tau)


    # Add all variables to var_dict by name.
    print(f"'N': {repr(N)},")
    print(f"'d': {repr(d)},")
    print(f"'Y': {repr(Y)},")
    print(f"'h': {repr(h)},")
    print(f"'dW': {repr(dW)},")
    print(f"'I': {repr(I)},")

    # Perform SRI2/SRS2 steps.
    for n in range(N - 1):
        Yn = Y[n, :]
        tn = t[n]
        tn1 = t[n + 1]
        sqrth = np.sqrt(h)

        # elementary increments
        Ik = dW[n, :]      # shape (d,)
        Iij = I[n, :, :]     # shape (d, d)
        assert Iij.shape == (d, d)

        # compute f*h
        fnh = f(Yn, tn) * h     # shape 

        # evaluate G at Yn
        Gn = G(Yn, tn)

        # sum1 = G·Iij / sqrt(h)
        sum1 = np.dot(Gn, Iij) / sqrth

        # H2 and H3 stages
        H20 = Yn + fnh
        H20b = np.reshape(H20, (d, 1))
        H2  = H20b + sum1
        H30 = Yn
        H3  = H20b - sum1

        # f at H20 and H20 at next time
        # TODO: Potentially intervene here.
        fn1h = f(H20, tn1) * h

        # base strong‐RK2 update
        Yn1 = Yn + 0.5 * (fnh + fn1h) + np.dot(Gn, Ik)

        print(f"'tn': {repr(tn)},")
        print(f"'tn1': {repr(tn1)},")
        print(f"'sqrth': {repr(sqrth)},")
        print(f"'Ik': {repr(Ik)},")
        print(f"'Iij': {repr(Iij)},")
        print(f"'fnh': {repr(fnh)},")
        print(f"'Gn': {repr(Gn)},")
        print(f"'sum1': {repr(sum1)},")
        print(f"'H2': {repr(H2)},")
        print(f"'H20': {repr(H20)},")
        print(f"'H20b': {repr(H20b)},")
        print(f"'H30': {repr(H30)},")
        print(f"'H3': {repr(H3)},")
        print(f"'fn1h': {repr(fn1h)},")
        print(f"'Yn1': {repr(Yn1)},")

        # corrector term
        for k in range(d):
            # TODO: Potentially apply intervention here.
            G_H2k = G(H2[:, k], tn1)    # shape (d,d)
            G_H3k = G(H3[:, k], tn1)

            Yn1 += 0.5 * sqrth * (G_H2k[:, k] - G_H3k[:, k])

        # apply intervention
        if intervention is not None:
            Yn1 = intervention(Yn1, tn1)

        Y[n + 1] = Yn1

    # --- 7) add measurement noise if requested ---
    if model.measurement_noise_std is not None:
        Y[1:] = model.add_measurement_noise(Y[1:], rng)

    return Y

In [34]:
def _Roessler2010_SRK2(
    f, G, y0, tspan, IJmethod, dW=None, IJ=None, generator=None):
    """Implements the Roessler2010 order 1.0 strong Stochastic Runge-Kutta
    algorithms SRI2 (for Ito equations) and SRS2 (for Stratonovich equations).

    Algorithms SRI2 and SRS2 are almost identical and have the same extended
    Butcher tableaus. The difference is that Ito repeateded integrals I_ij are
    replaced by Stratonovich repeated integrals J_ij when integrating a
    Stratonovich equation (Theorem 6.2 in Roessler2010).

    Args:
      f: A function f(y, t) returning an array of shape (d,)
      G: Either a function G(y, t) that returns an array of shape (d, m),
         or a list of m functions g(y, t) each returning an array shape (d,).
      y0: array of shape (d,) giving the initial state
      tspan (array): Sequence of equally spaced time points
      IJmethod (callable): which function to use to generate repeated
        integrals. N.B. for an Ito equation, must use an Ito version here
        (either Ikpw or Iwik). For a Stratonovich equation, must use a
        Stratonovich version here (Jkpw or Jwik).
      dW: optional array of shape (len(tspan)-1, d).
      IJ: optional array of shape (len(tspan)-1, m, m).
        Optional arguments dW and IJ are for advanced use, if you want to
        use a specific realization of the d independent Wiener processes and
        their multiple integrals at each time step. If not provided, suitable
        values will be generated randomly.
      generator (numpy.random.Generator, optional) Random number generator
        instance to use. If omitted, a new default_rng will be instantiated.

    Returns:
      y: array, with shape (len(tspan), len(y0))

    Raises:
      SDEValueError

    See also:
      A. Roessler (2010) Runge-Kutta Methods for the Strong Approximation of
        Solutions of Stochastic Differential Equations
    """
    (d, m, f, G, y0, tspan, dW, IJ) = _check_args(f, G, y0, tspan, dW, IJ)
    if generator is None and (dW is None or IJ is None):
        generator = np.random.default_rng()
    have_separate_g = (not callable(G)) # if G is given as m separate functions
    N = len(tspan)
    h = (tspan[N-1] - tspan[0])/(N - 1) # assuming equal time steps
    if dW is None:
        # pre-generate Wiener increments (for m independent Wiener processes):
        dW = deltaW(N - 1, m, h, generator) # shape (N, m)
    if IJ is None:
        # pre-generate repeated stochastic integrals for each time step.
        # Must give I_ij for the Ito case or J_ij for the Stratonovich case:
        __, I = IJmethod(dW, h, generator=generator) # shape (N, m, m)
    else:
        I = IJ
    # allocate space for result
    y = np.zeros((N, d), dtype=y0.dtype)
    y[0] = y0;
    Gn = np.zeros((d, m), dtype=y.dtype)
    print(f"'N': {repr(N)},")
    print(f"'d': {repr(d)},")
    print(f"'Y': {repr(y)},")
    print(f"'h': {repr(h)},")
    print(f"'dW': {repr(dW)},")
    print(f"'I': {repr(I)},")

    for n in range(0, N-1):
        tn = tspan[n]
        tn1 = tspan[n+1]
        h = tn1 - tn
        sqrth = np.sqrt(h)
        Yn = y[n] # shape (d,)
        Ik = dW[n,:] # shape (m,)
        Iij = I[n,:,:] # shape (m, m)
        fnh = f(Yn, tn)*h # shape (d,)
        if have_separate_g:
            for k in range(0, m):
                Gn[:,k] = G[k](Yn, tn)
        else:
            Gn = G(Yn, tn)
        sum1 = np.dot(Gn, Iij)/sqrth # shape (d, m)
        H20 = Yn + fnh # shape (d,)
        H20b = np.reshape(H20, (d, 1))
        H2 = H20b + sum1 # shape (d, m)
        H30 = Yn
        H3 = H20b - sum1
        fn1h = f(H20, tn1)*h
        Yn1 = Yn + 0.5*(fnh + fn1h) + np.dot(Gn, Ik)
        print(f"'tn': {repr(tn)},")
        print(f"'tn1': {repr(tn1)},")
        print(f"'sqrth': {repr(sqrth)},")
        print(f"'Ik': {repr(Ik)},")
        print(f"'Iij': {repr(Iij)},")
        print(f"'fnh': {repr(fnh)},")
        print(f"'Gn': {repr(Gn)},")
        print(f"'sum1': {repr(sum1)},")
        print(f"'H2': {repr(H2)},")
        print(f"'H20': {repr(H20)},")
        print(f"'H20b': {repr(H20b)},")
        print(f"'H30': {repr(H30)},")
        print(f"'H3': {repr(H3)},")
        print(f"'fn1h': {repr(fn1h)},")
        print(f"'Yn1': {repr(Yn1)},")

        if have_separate_g:
            for k in range(0, m):
                Yn1 += 0.5*sqrth*(G[k](H2[:,k], tn1) - G[k](H3[:,k], tn1))
        else:
            for k in range(0, m):
                Yn1 += 0.5*sqrth*(G(H2[:,k], tn1)[:,k] - G(H3[:,k], tn1)[:,k])
        y[n+1] = Yn1
    return y

In [40]:
seed = 11
rng = np.random.default_rng(seed)
n = 3
theta = rng.random((n, n)) - 0.5
mu = np.ones(n)
sigma = rng.random((n, n))- 0.5

model = interfere.dynamics.OrnsteinUhlenbeck(theta, mu, sigma)

x0 = rng.random(n)
tspan = np.arange(0, 0.2, 0.1)
dt = (tspan[-1] - tspan[0]) / len(tspan)

# Check that using the same generator corresponds exactly with sdeint
seed = 11
rng = np.random.default_rng(seed)
Xtrue = _Roessler2010_SRK2(
    model.drift, model.noise, x0, tspan, IJmethod=Ikpw ,generator=rng)

seed = 11
rng = np.random.default_rng(seed)
Xsim = _simulate_sri2(model, tspan, x0.reshape((1, -1)), rng=rng)

'N': 2,
'd': 3,
'Y': array([[0.81673644, 0.54907527, 0.98091364],
       [0.        , 0.        , 0.        ]]),
'h': 0.1,
'dW': array([[0.0108127 , 0.42998993, 0.38729081]]),
'I': array([[[-0.04994154, -0.0440523 , -0.0466538 ],
        [ 0.04870166,  0.04244567,  0.1120416 ],
        [ 0.05084146,  0.05448955,  0.02499709]]]),
'tn': 0.0,
'tn1': 0.1,
'sqrth': 0.31622776601683794,
'Ik': array([0.0108127 , 0.42998993, 0.38729081]),
'Iij': array([[-0.04994154, -0.0440523 , -0.0466538 ],
       [ 0.04870166,  0.04244567,  0.1120416 ],
       [ 0.05084146,  0.05448955,  0.02499709]]),
'fnh': array([-0.00664579, -0.023696  , -0.02371134]),
'Gn': array([[ 0.12188359, -0.13100688,  0.01139002],
       [ 0.16284295, -0.22469118, -0.36203193],
       [ 0.28803959,  0.17036058,  0.01238231]]),
'sum1': array([[-0.03759386, -0.03260084, -0.06349801],
       [-0.11852752, -0.11522622, -0.13225197],
       [-0.01726213, -0.01512528,  0.01884355]]),
'H2': array([[0.77249678, 0.7774898 , 0.74659263],


In [41]:
from numpy import array

true_vars = {
'N': 2,
'd': 3,
'Y': array([[0.81673644, 0.54907527, 0.98091364],
       [0.        , 0.        , 0.        ]]),
'h': 0.1,
'dW': array([[0.0108127 , 0.42998993, 0.38729081]]),
'I': array([[[-0.04994154, -0.0440523 , -0.0466538 ],
        [ 0.04870166,  0.04244567,  0.1120416 ],
        [ 0.05084146,  0.05448955,  0.02499709]]]),
'tn': 0.0,
'tn1': 0.1,
'sqrth': 0.31622776601683794,
'Ik': array([0.0108127 , 0.42998993, 0.38729081]),
'Iij': array([[-0.04994154, -0.0440523 , -0.0466538 ],
       [ 0.04870166,  0.04244567,  0.1120416 ],
       [ 0.05084146,  0.05448955,  0.02499709]]),
'fnh': array([-0.00664579, -0.023696  , -0.02371134]),
'Gn': array([[ 0.12188359, -0.13100688,  0.01139002],
       [ 0.16284295, -0.22469118, -0.36203193],
       [ 0.28803959,  0.17036058,  0.01238231]]),
'sum1': array([[-0.03759386, -0.03260084, -0.06349801],
       [-0.11852752, -0.11522622, -0.13225197],
       [-0.01726213, -0.01512528,  0.01884355]]),
'H2': array([[0.77249678, 0.7774898 , 0.74659263],
       [0.40685175, 0.41015305, 0.3931273 ],
       [0.93994017, 0.94207702, 0.97604585]]),
'H20': array([0.81009064, 0.52537927, 0.9572023 ]),
'H20b': array([[0.81009064],
       [0.52537927],
       [0.9572023 ]]),
'H30': array([0.81673644, 0.54907527, 0.98091364]),
'H3': array([[0.8476845 , 0.84269148, 0.87358866],
       [0.64390679, 0.64060549, 0.65763124],
       [0.97446443, 0.97232758, 0.93835875]]),
'fn1h': array([-0.00665368, -0.02382815, -0.02381107]),
'Yn1': array([0.7594842 , 0.29024738, 1.03831581]),
}

my_vars = {
'N': 2,
'd': 3,
'Y': array([[0.81673644, 0.54907527, 0.98091364],
       [0.        , 0.        , 0.        ]]),
'h': 0.1,
'dW': array([[0.0108127 , 0.42998993, 0.38729081]]),
'I': array([[[-0.04994154, -0.0440523 , -0.0466538 ],
        [ 0.04870166,  0.04244567,  0.1120416 ],
        [ 0.05084146,  0.05448955,  0.02499709]]]),
'tn': 0.0,
'tn1': 0.1,
'sqrth': 0.31622776601683794,
'Ik': array([0.0108127 , 0.42998993, 0.38729081]),
'Iij': array([[-0.04994154, -0.0440523 , -0.0466538 ],
       [ 0.04870166,  0.04244567,  0.1120416 ],
       [ 0.05084146,  0.05448955,  0.02499709]]),
'fnh': array([-0.00664579, -0.023696  , -0.02371134]),
'Gn': array([[ 0.12188359, -0.13100688,  0.01139002],
       [ 0.16284295, -0.22469118, -0.36203193],
       [ 0.28803959,  0.17036058,  0.01238231]]),
'sum1': array([[-0.03759386, -0.03260084, -0.06349801],
       [-0.11852752, -0.11522622, -0.13225197],
       [-0.01726213, -0.01512528,  0.01884355]]),
'H2': array([[0.77249678, 0.7774898 , 0.74659263],
       [0.40685175, 0.41015305, 0.3931273 ],
       [0.93994017, 0.94207702, 0.97604585]]),
'H20': array([0.81009064, 0.52537927, 0.9572023 ]),
'H20b': array([[0.81009064],
       [0.52537927],
       [0.9572023 ]]),
'H30': array([0.81673644, 0.54907527, 0.98091364]),
'H3': array([[0.8476845 , 0.84269148, 0.87358866],
       [0.64390679, 0.64060549, 0.65763124],
       [0.97446443, 0.97232758, 0.93835875]]),
'fn1h': array([-0.00665368, -0.02382815, -0.02381107]),
'Yn1': array([0.7594842 , 0.29024738, 1.03831581]),
}

In [42]:

x0

array([0.81673644, 0.54907527, 0.98091364])

In [43]:
for key in true_vars.keys():
    if not np.all(true_vars[key] == my_vars[key]):
        print(
            f"Variable mismatch: {key}"
            f"\n\tTrue: {true_vars[key]}"
            f"\n\tMine: {my_vars[key]}"
        )

In [ ]:
seed = 11
rng = np.random.default_rng(seed)
n = 3
theta = rng.random((n, n)) - 0.5
mu = np.ones(n)
sigma = rng.random((n, n))- 0.5

model = interfere.dynamics.OrnsteinUhlenbeck(theta, mu, sigma)

x0 = rng.random(n)
tspan = np.linspace(0, 10, 1000)
dt = (tspan[-1] - tspan[0]) / len(tspan)

# Check that using the same generator corresponds exactly with sdeint
seed = 11
rng = np.random.default_rng(seed)
Xtrue = sdeint.itoSRI2(model.drift, model.noise, x0, tspan, generator=rng)

seed = 11
rng = np.random.default_rng(seed)
Xsim = model._simulate_sdeint_sri2(tspan, x0, rng=rng)

assert np.mean((Xtrue - Xsim) ** 2) < 0.01

# Initialize the Weiner increments
dW = np.random.normal(0, np.sqrt(dt), (len(tspan) - 1, n))
_, I = sdeint.wiener.Ikpw(dW, h=dt)

# Check that the model.simulate_sri2 integrator is correct
Xtrue = sdeint.itoSRI2(model.drift, model.noise, x0, tspan, dW=dW, I=I)
Xsim = model._simulate_sdeint_sri2(tspan, x0, dW=dW, I=I)
assert np.mean((Xtrue - Xsim) ** 2) < 0.01

# Construct parameters of the true intervened system
theta_perf_inter = model.theta.copy()
sigma_perf_inter = model.sigma.copy()

theta_perf_inter[0, :] = 0
sigma_perf_inter[0, :] = 0

# True perfect intervention noise and drift functions
perf_inter_drift = lambda x, t: theta_perf_inter @ (model.mu - x)
perf_inter_noise = lambda x, t: sigma_perf_inter

# Make the intervention function
interv_idx = 0
interv_const = 1.0
intervention = interfere.perfect_intervention(interv_idx, interv_const)

# Compare the true perfect intervention system to the true one.
rng = np.random.default_rng(seed)
X_perf_inter = sdeint.itoSRI2(
    perf_inter_drift,
    perf_inter_noise,
    intervention(x0, 0),
    tspan,
    generator=rng,
    dW=dW,
    I=I
)

rng = np.random.default_rng(seed)
X_perf_inter_sim = model._simulate_sdeint_sri2(
    tspan,
    x0,
    intervention=intervention,
    rng=rng,
    dW=dW,
    I=I
)

# Check that the intervened variable is constant
assert np.all(X_perf_inter_sim[:, interv_idx] == interv_const)

# Check that the simulations match
assert np.mean((X_perf_inter - X_perf_inter_sim) ** 2) < 0.01